# Pertemuan 5 — Divide and Conquer II
Karatsuba, Strassen (+ titik impas), dan Closest Pair. **[Modern]** catatan AlphaTensor/AlphaDev.

In [ ]:
import math, random, time
import numpy as np
import matplotlib.pyplot as plt
random.seed(5); np.random.seed(5)

## 1. Karatsuba vs perkalian sekolah (T(n)=3T(n/2)+n → n^1.585)
Kita hitung banyaknya perkalian "limb" 16-bit (analogi perkalian satu digit).

In [ ]:
B = 16; MASK = (1 << B) - 1
def schoolbook(x, y, cnt):
    xs, ys = [], []
    while x: xs.append(x & MASK); x >>= B
    while y: ys.append(y & MASK); y >>= B
    res = 0
    for i, a in enumerate(xs):
        for j, b in enumerate(ys):
            cnt[0] += 1; res += (a * b) << (B * (i + j))
    return res

def karatsuba(x, y, cnt):
    if x <= MASK or y <= MASK:
        cnt[0] += 1; return x * y
    m = max(x.bit_length(), y.bit_length()) // 2
    xh, xl = x >> m, x & ((1 << m) - 1); yh, yl = y >> m, y & ((1 << m) - 1)
    z0 = karatsuba(xl, yl, cnt); z2 = karatsuba(xh, yh, cnt)
    z1 = karatsuba(xl + xh, yl + yh, cnt) - z0 - z2
    return (z2 << (2 * m)) + (z1 << m) + z0

bits = [256, 512, 1024, 2048, 4096, 8192]
cs, ck = [], []
for nb in bits:
    x, y = random.getrandbits(nb) | 1 << (nb-1), random.getrandbits(nb) | 1 << (nb-1)
    a, b = [0], [0]
    assert schoolbook(x, y, a) == karatsuba(x, y, b) == x * y
    cs.append(a[0]); ck.append(b[0])
print("bits  schoolbook  karatsuba")
for nb, a_, b_ in zip(bits, cs, ck): print(f"{nb:>5} {a_:>10} {b_:>10}")
print("kemiringan log-log: schoolbook ≈ %.2f , karatsuba ≈ %.2f  (teori 2 dan 1.585)" %
      (np.polyfit(np.log(bits), np.log(cs), 1)[0], np.polyfit(np.log(bits), np.log(ck), 1)[0]))

## 2. Strassen: 7 perkalian, bukan 8

In [ ]:
def strassen(A, Bm, cnt, cutoff=1):
    n = A.shape[0]
    if n <= cutoff:
        cnt[0] += n**3; return A @ Bm
    h = n // 2
    A11, A12, A21, A22 = A[:h,:h], A[:h,h:], A[h:,:h], A[h:,h:]
    B11, B12, B21, B22 = Bm[:h,:h], Bm[:h,h:], Bm[h:,:h], Bm[h:,h:]
    M1 = strassen(A11 + A22, B11 + B22, cnt, cutoff); M2 = strassen(A21 + A22, B11, cnt, cutoff)
    M3 = strassen(A11, B12 - B22, cnt, cutoff);       M4 = strassen(A22, B21 - B11, cnt, cutoff)
    M5 = strassen(A11 + A12, B22, cnt, cutoff);       M6 = strassen(A21 - A11, B11 + B12, cnt, cutoff)
    M7 = strassen(A12 - A22, B21 + B22, cnt, cutoff)
    C = np.empty_like(A)
    C[:h,:h] = M1 + M4 - M5 + M7; C[:h,h:] = M3 + M5; C[h:,:h] = M2 + M4; C[h:,h:] = M1 - M2 + M3 + M6
    return C

for n in [2, 4, 8, 16, 32, 64]:
    A, Bm = np.random.randint(-5, 5, (n, n)), np.random.randint(-5, 5, (n, n)); c = [0]
    assert np.array_equal(strassen(A, Bm, c), A @ Bm)
    print(f"n={n:>3}  perkalian Strassen={c[0]:>7}  naif n³={n**3:>7}  rasio={c[0]/n**3:.3f}")

### Titik impas (crossover) teoritis
Hitung total operasi aritmetika (kali + tambah): naif = 2n³ − n²; Strassen = 7·S(n/2) + 18·(n/2)². Dengan *cutoff* n₀, Strassen berhenti rekursi dan memakai metode naif.

In [ ]:
def naive_ops(n): return 2 * n**3 - n**2
def strassen_ops(n, n0):
    if n <= n0: return naive_ops(n)
    return 7 * strassen_ops(n // 2, n0) + 18 * (n // 2) ** 2

print(f"{'n':>6} " + " ".join(f"n0={c:>3}" .rjust(12) for c in (1, 16, 64)) + "   naif")
for k in range(3, 13):
    n = 2**k
    print(f"{n:>6} " + " ".join(f"{strassen_ops(n, c):>12.3e}" for c in (1, 16, 64)) + f" {naive_ops(n):>12.3e}")
cross = next(2**k for k in range(1, 20) if strassen_ops(2**k, 1) < naive_ops(2**k))
print("\nStrassen murni (n0=1) mengalahkan naif mulai n =", cross, "(secara jumlah operasi)")

**Empiris:** dengan `numpy` sebagai kasus dasar, `A @ B` (BLAS) sangat teroptimasi sehingga Strassen jarang menang di ukuran kecil — inilah pelajaran *crossover* di praktik.

In [ ]:
for n in [128, 256, 512]:
    A, Bm = np.random.rand(n, n), np.random.rand(n, n)
    t = time.perf_counter(); A @ Bm; t1 = time.perf_counter() - t
    t = time.perf_counter(); strassen(A, Bm, [0], cutoff=64); t2 = time.perf_counter() - t
    print(f"n={n}: BLAS={t1*1000:.2f} ms | Strassen(cutoff 64)={t2*1000:.2f} ms")

## 3. Closest Pair of Points — O(n log² n) (versi sederhana) vs O(n²)

In [ ]:
def brute_closest(P):
    return min(math.dist(P[i], P[j]) for i in range(len(P)) for j in range(i + 1, len(P)))

def closest_pair(P):
    def rec(px):
        n = len(px)
        if n <= 3: return brute_closest(px)
        mid = n // 2; midx = px[mid][0]
        d = min(rec(px[:mid]), rec(px[mid:]))
        strip = sorted((p for p in px if abs(p[0] - midx) < d), key=lambda p: p[1])
        for i in range(len(strip)):
            for j in range(i + 1, min(i + 8, len(strip))):
                d = min(d, math.dist(strip[i], strip[j]))
        return d
    return rec(sorted(P))

for n in [200, 800, 3200]:
    P = [(random.random(), random.random()) for _ in range(n)]
    t = time.perf_counter(); d1 = closest_pair(P); t1 = time.perf_counter() - t
    if n <= 800:
        t = time.perf_counter(); d2 = brute_closest(P); t2 = time.perf_counter() - t
        print(f"n={n}: D&C {t1:.4f}s | brute {t2:.4f}s | sama? {abs(d1-d2)<1e-12}")
    else: print(f"n={n}: D&C {t1:.4f}s (brute force dilewati)")

## 4. [Modern] AI menemukan algoritma
- **AlphaTensor** (Nature, 2022): RL mencari dekomposisi tensor perkalian matriks; melaporkan algoritma dengan lebih sedikit perkalian dibanding Strassen untuk ukuran tertentu (mis. matriks 4×4 pada aritmetika modulo 2: 47 perkalian vs 49 dari Strassen dua tingkat).
- **AlphaDev** (Nature, 2023): menemukan rutin sorting assembly untuk urutan pendek yang kemudian masuk ke pustaka standar LLVM libc++.

Ide kuncinya sama dengan kuliah ini: mengurangi **jumlah operasi dominan** — hanya saja pencariannya dibantu AI. Baca makalah aslinya untuk detail dan syarat (field aritmetika, ukuran).

## 5. Mini Proyek / Tugas
1. Tambahkan penanganan matriks non-pangkat-dua (padding) pada `strassen`.
2. Cari **cutoff optimal** `n0` secara empiris untuk mesin Colab Anda (grafik waktu vs cutoff).
3. Plot kemiringan log-log waktu Strassen vs naif (kode Python murni, tanpa BLAS) dan bandingkan dengan 2.807 vs 3.
4. Lengkapi laporan: titik impas teoritis vs empiris, jelaskan kenapa berbeda.

In [ ]:
def strassen_padded(A, Bm):
    pass  # TODO